# 08쪽 · 도구 요청과 실제 실행

## 준비
아래 셀은 파일 경로와 `.env` 로더를 준비합니다. 한 번 실행하고 넘어가세요.

In [ ]:
from pathlib import Path
import sys

# 노트북을 연 위치가 ipynb 폴더이면 한 단계 위로 이동합니다.
ROOT = Path.cwd()

if ROOT.name == "ipynb":
    ROOT = ROOT.parent

if not (ROOT / "lab" / "config.py").exists():
    raise RuntimeError("Langgraph_6_Week_Code 또는 ipynb 폴더에서 실행하세요.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lab.config import make_llm, load_settings

## 1. 도구 함수 정의

PPT에 있던 덧셈과 곱셈 함수입니다.

In [ ]:
def add_numbers(a: int, b: int) -> int:
    """두 정수 a와 b를 더합니다."""
    return a + b


def multiply_numbers(a: int, b: int) -> int:
    """두 정수 a와 b를 곱합니다."""
    return a * b

## 2. 모델에 도구 연결

아직 함수를 실행하지 않습니다.

In [ ]:
llm = make_llm()

tools = [add_numbers, multiply_numbers]
tool_llm = llm.bind_tools(
    tools,
    parallel_tool_calls=False,
)

## 3. 모델에 덧셈 요청

먼저 모델이 반환한 도구 요청만 봅니다.

In [ ]:
from langchain_core.messages import HumanMessage

message = HumanMessage(content="10과 20을 더해줘. 덧셈 도구를 사용해줘.")
response = tool_llm.invoke([message])

response.pretty_print()

**결과 해설**

AIMessage의 Tool Calls에서 add_numbers와 a=10, b=20을 확인합니다.

## 4. ToolNode로 실행

앞 셀의 요청을 실제 함수에 전달합니다.

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

if response.tool_calls:
    result = tool_node.invoke({
        "messages": [response],
    })

    for message in result["messages"]:
        message.pretty_print()
else:
    print("모델이 도구를 요청하지 않았습니다. 요청 셀을 다시 확인하세요.")

**결과 해설**

덧셈 요청이 있었다면 ToolMessage에 계산 결과 30이 나옵니다. AIMessage는 요청, ToolMessage는 실행 결과입니다.